# Does a farming game get supply and demand right?

Kaggriculture is a simulation competition where your farm sells into a shared
market, and the market pushes the price down as you sell. The engine's pricing rule
is short, public, and completely explicit — which makes it possible to ask a question
you cannot usually ask of a game: **is its supply-and-demand curve anywhere near a
real one?**

To answer it we need a real market with both quantity and price, over enough years to
measure the relationship. US honey is a good one: fifteen years, forty-odd states,
production and price per pound in the same table.

The plan:

1. The naive answer, and why it is wrong by a factor of sixteen
2. The panel answer, with year and state fixed effects
3. The elasticity the game's own source code implies
4. The verdict, as a number

Nothing here is competition advice — it is a comparison between a rule in a source
file and a regression on public data.

> **Version pin.** The competition ladder runs `kaggle-environments` **1.32.4** and
> its market parameters differ from older builds, so we pin it rather than take
> whatever the notebook image ships.

In [ ]:
%pip install -q "kaggle-environments==1.32.4"

In [ ]:
import glob, math, os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8, 4.2), "axes.grid": True, "grid.alpha": .25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

from kaggle_environments.envs.kaggriculture import kaggriculture as K
try:
    from importlib.metadata import version as _pkg_version
    print("kaggle-environments", _pkg_version("kaggle-environments"))
except Exception:
    pass

SRC_LINES = open(K.__file__, encoding="utf-8").read().split("\n")
import inspect

def show_func(obj, title=None):
    "Print a referee function's real source, with its real line numbers."
    lines, start = inspect.getsourcelines(obj)
    print(f"--- {title or obj.__name__}  [lines {start}-{start+len(lines)-1}] ---")
    for i, line in enumerate(lines):
        print(f"{start + i:4d} | {line.rstrip()}")

def show_block(anchor, after=0, before=0, title=None):
    "Print lines around the first line containing `anchor` (line numbers move\n"
    "between releases, so we never hard-code a range)."
    hits = [i for i, l in enumerate(SRC_LINES) if anchor in l]
    if not hits:
        print(f"!! anchor not found in this version: {anchor!r}")
        return
    i = hits[0]
    lo, hi = max(0, i - before), min(len(SRC_LINES), i + after + 1)
    print(f"--- {title or anchor}  [lines {lo+1}-{hi}] ---")
    for j in range(lo, hi):
        print(f"{j + 1:4d} | {SRC_LINES[j].rstrip()}")

In [ ]:
def find_csv(name):
    for pattern in (f"/kaggle/input/**/{name}", f"../input/**/{name}", name):
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            return hits[0]
    raise SystemExit(f"{name} not found — attach jessicali9530/honey-production")

honey = pd.read_csv(find_csv("honeyproduction.csv"))
honey = honey[(honey.totalprod > 0) & (honey.priceperlb > 0)].copy()
print(f"{len(honey)} rows | {honey.state.nunique()} states | "
      f"{honey.year.min()}-{honey.year.max()}")
honey.head()

## 1. The naive answer

Add up production across states, average the price, and regress one log on the other.
This is the analysis almost everyone runs first.

In [ ]:
nat = honey.groupby("year").agg(supply=("totalprod", "sum"),
                                price=("priceperlb", "mean")).reset_index()
slope_naive = np.polyfit(np.log(nat.supply), np.log(nat.price), 1)[0]
r_naive = np.corrcoef(np.log(nat.supply), np.log(nat.price))[0, 1]
print(f"naive elasticity: {slope_naive:.2f}   (r = {r_naive:.2f}, n = {len(nat)})")
nat.assign(supply_Mlb=(nat.supply / 1e6).round(1))[["year", "supply_Mlb", "price"]]

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(nat.year, nat.supply / 1e6, "o-", color="#2b6cb0")
a1.set_title("US honey supply"); a1.set_ylabel("million lb"); a1.set_xlabel("year")
a2.plot(nat.year, nat.price, "o-", color="#c05621")
a2.set_title("average price"); a2.set_ylabel("$/lb"); a2.set_xlabel("year")
plt.tight_layout(); plt.show()

An elasticity of about **−2.1**: supply down 10%, price up 21%. It looks like a
textbook demand curve, and it is almost certainly wrong.

Look at the two panels again. Supply falls in nearly every year. Price rises in nearly
every year. Regressing one trending series on another recovers the ratio of their
trends, not a demand relationship — and there is plenty else moving over 1998–2012 to
push both, including colony collapse disorder on the supply side and ordinary
inflation on the price side.

The test for this is simple: if each series is well explained by *time alone*, the
correlation between them carries little independent information.

In [ ]:
rows = []
for name, s in [("log supply", np.log(nat.supply)), ("log price", np.log(nat.price))]:
    slope = np.polyfit(nat.year, s, 1)[0]
    r = np.corrcoef(nat.year, s)[0, 1]
    rows.append({"series": name, "trend per year": round(slope, 4),
                 "r with year": round(r, 2), "R2 on year alone": round(r ** 2, 2)})
pd.DataFrame(rows).set_index("series")

Both series are three-quarters explained by the calendar. That is the signature of a
spurious regression, and it means the −2.1 has to be thrown away rather than
interpreted.

## 2. The panel answer

The data has a second dimension we have been discarding: forty-odd states, each with
its own production and its own price, every year. That lets us ask a much narrower
question — **within a single year, do states that produced more get a lower price?** —
which no national trend can contaminate.

That is a fixed-effects regression. Below is a plain OLS with dummy variables, written
out rather than imported, so the estimate is inspectable.

In [ ]:
def ols_fe(df, y, x, absorb=()):
    "log-log OLS with dummy fixed effects. Returns (beta, se, R2, n)."
    Y = np.log(df[y].values)
    cols = [np.ones(len(df)), np.log(df[x].values)]
    for col in absorb:
        codes = pd.Categorical(df[col]).codes
        for k in range(1, codes.max() + 1):     # first level is the reference
            cols.append((codes == k).astype(float))
    X = np.column_stack(cols)
    beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
    resid = Y - X @ beta
    dof = len(df) - X.shape[1]
    s2 = resid @ resid / dof
    se = math.sqrt(s2 * np.linalg.pinv(X.T @ X)[1, 1])
    r2 = 1 - (resid @ resid) / ((Y - Y.mean()) @ (Y - Y.mean()))
    return beta[1], se, r2, len(df)

specs = [("pooled (no fixed effects)", ()),
         ("year fixed effects", ("year",)),
         ("year + state fixed effects", ("year", "state"))]
out = []
for label, absorb in specs:
    b, se, r2, n = ols_fe(honey, "priceperlb", "totalprod", absorb)
    out.append({"specification": label, "elasticity": round(b, 3),
                "std err": round(se, 3),
                "95% CI": f"[{b - 1.96*se:+.2f}, {b + 1.96*se:+.2f}]",
                "R2": round(r2, 2), "n": n})
panel = pd.DataFrame(out).set_index("specification")
panel

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
labels = ["naive\n(national, no FE)"] + [s[0].replace(" ", "\n", 1) for s in specs]
vals = [slope_naive] + [float(panel.loc[s[0], "elasticity"]) for s in specs]
errs = [0] + [1.96 * float(panel.loc[s[0], "std err"]) for s in specs]
ax.bar(labels, vals, yerr=errs, capsize=4,
       color=["#a0aec0", "#90cdf4", "#2b6cb0", "#1a365d"])
ax.axhline(0, color="black", lw=1)
ax.set_ylabel("price elasticity of supply\n(d log price / d log quantity)")
ax.set_title("The naive estimate is 16x the within-year one")
plt.tight_layout(); plt.show()

The honest numbers:

* **Year fixed effects: about −0.13.** Comparing states inside the same year, a state
  producing 10% more honey sells it for roughly 1.3% less. Tight interval, and the
  specification explains ~79% of the variation.
* **Year *and* state fixed effects: about −0.05.** Once each state is allowed its own
  price level — some states genuinely produce a more valuable product — most of the
  remaining relationship goes away.

**And that second interval reaches zero.** Its upper bound rounds to −0.00, so at this
specification we cannot reject the hypothesis that there is *no* within-state
supply-price relationship at all. That is worth saying plainly rather than quoting
−0.05 as if it were established: the honest reading is that the true elasticity is
somewhere between "small" and "not distinguishable from nothing".

So the real elasticity is small either way. The naive −2.1 overstated it roughly
**sixteenfold**. We carry the year-FE figure of −0.13 forward as the *generous*
benchmark — generous because the stricter specification is flatter still, which can
only make the game look steeper by comparison.

## 3. What the game's source implies

Now the other side of the comparison. Here is the entire pricing rule.

In [ ]:
show_func(K.market_price)
show_block("price(inv) = base", after=11, before=1,
           title="the model, in the source's own words")

Price is a function of market inventory: a base price, a neutral level `I0`, and a
curve that pulls the price down as inventory rises above it. The amplitude is
calibrated so that adding `T` units moves the price by `target × base`, and the source
tells us exactly what `T` means — *the production capacity of one 5×5 field over a
24-day window*. That gives us a natural unit: **one field-season of supply.**

Two details make a finite difference useless here: `market_price` rounds to an integer
and floors at `PRICE_FLOOR`. So we differentiate the underlying formula instead.

### What is and is not being compared

The two quantities are not the same thing, so it is worth being precise about what
carries across.

| | US honey | Kaggriculture |
|---|---|---|
| quantity | pounds produced in a year, summed over states | units sold into the shared market |
| price | dollars per pound, annual average | coins per unit, re-quoted every unit |
| market size | ~180 million lb a year | neutral inventory `I0` = 10,000 units |
| natural unit of supply | a state's annual crop | `T`, one 5×5 field over 24 days (100–450 units) |

**Absolute scale does not transfer, and we never use it.** An elasticity is a ratio of
percentage changes, so it is dimensionless: "supply up 10%, price down 1.3%" means the
same thing whether the market is measured in pounds or coins, in millions or hundreds.
That is the whole reason this comparison is possible at all.

What does *not* carry across is the range over which each was measured. The honey
figure comes from year-to-year variation of roughly ±10% around the national crop.
The game's elasticity is not a constant, so quoting a single number for it would be
meaningless — instead we evaluate it at several fractions of one field-season and let
the reader see where it stops resembling the real one.

In [ ]:
SHAPES = {
    "linear": (lambda x: x,               lambda x: 1.0),
    "sq":     (lambda x: x * x,           lambda x: 2 * x),
    "sqrt":   (lambda x: math.sqrt(x),    lambda x: 1 / (2 * math.sqrt(x))),
    "log":    (lambda x: math.log(1 + x), lambda x: 1 / (1 + x)),
    "log10":  (lambda x: math.log10(1+x), lambda x: 1 / ((1+x) * math.log(10))),
}

def game_elasticity(item, q):
    "d log P / d log Q for q units sold above neutral, before rounding/flooring."
    p = K.MARKET_PARAMS[item]
    f, df = SHAPES[p["above_func"]]
    amp = p["above_target"] * p["base"] / f(p["T"])
    P = p["base"] - amp * f(q)
    return np.nan if P <= 0 else (-amp * df(q)) * q / P

def floor_at(item, cap=5000):
    "Units of supply at which the quoted price reaches the floor."
    for q in range(1, cap + 1):
        if K.market_price(item, K.MARKET_I0 + q) <= K.PRICE_FLOOR:
            return q
    return None

ITEMS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL"]
rows = []
for it in ITEMS:
    T = K.MARKET_PARAMS[it]["T"]
    rows.append({"product": it, "T (one field-season)": T,
                 "base price": K.MARKET_PARAMS[it]["base"],
                 "@ T/10": round(game_elasticity(it, T * .10), 2),
                 "@ T/4": round(game_elasticity(it, T * .25), 2),
                 "@ T/2": round(game_elasticity(it, T * .50), 2),
                 "floor hit at": floor_at(it)})
game = pd.DataFrame(rows).set_index("product")
game

In [ ]:
fig, ax = plt.subplots()
qs = np.linspace(0.02, 0.6, 120)
for it, colour in [("MELON", "#c05621"), ("STRAWBERRY", "#b83280"),
                   ("TOMATO", "#2b6cb0"), ("WHEAT", "#2f855a")]:
    T = K.MARKET_PARAMS[it]["T"]
    ax.plot(qs, [game_elasticity(it, T * f) for f in qs], label=it,
            color=colour, lw=2)
ax.axhline(-0.128, ls="--", color="black", lw=1.4,
           label="US honey, year FE (-0.13)")
ax.set_xlabel("supply sold, as a fraction of one field-season (T)")
ax.set_ylabel("elasticity (d log P / d log Q)")
ax.set_ylim(-5, 0.2)
ax.set_title("The game's elasticity is not a constant — it detonates")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

## 4. The verdict

A real market has roughly one elasticity. The game has a different one at every
volume, so the answer depends on how much you sell.

In [ ]:
HONEY_FE = float(panel.loc["year fixed effects", "elasticity"])
HONEY_STRICT = float(panel.loc["year + state fixed effects", "elasticity"])

verdict = []
for label, frac in [("a tenth of a field-season", .10),
                    ("a quarter", .25), ("half", .50)]:
    es = [game_elasticity(it, K.MARKET_PARAMS[it]["T"] * frac) for it in ITEMS]
    med = float(np.nanmedian(es))
    verdict.append({"volume sold": label, "game (median)": round(med, 2),
                    "US honey (year FE)": HONEY_FE,
                    "game / honey": round(med / HONEY_FE, 1)})
pd.DataFrame(verdict).set_index("volume sold")

In [ ]:
n_floor = sum(1 for it in ITEMS
              if (floor_at(it) or 10 ** 9) <= K.MARKET_PARAMS[it]["T"])
print(f"products whose price hits the floor within one field-season: "
      f"{n_floor} of {len(ITEMS)}")
for it in ITEMS:
    q, T = floor_at(it), K.MARKET_PARAMS[it]["T"]
    if q and q <= T:
        print(f"   {it:<11} floor at {q} units = {100*q/T:.0f}% of T")

**The answer is: both, depending on volume — and the crossover is early.**

* **Sell a little and the game is close to right.** At a tenth of a field-season the
  median product's elasticity is about **−0.10**, against **−0.13** for US honey.
  That is the same order of magnitude, and marginally *flatter* than reality.
* **Sell a realistic amount and it wildly exaggerates.** By a quarter of a
  field-season the median is roughly **−0.4**, three times the real figure. By half,
  the median is around **−2.3**, roughly eighteen times reality — and for melon
  specifically it passes **−18**.
* **Then it stops being a market at all.** Four of the eight products — strawberry,
  melon, milk and wool — hit the hard price floor of 1 coin at just over half a
  field-season. Real prices do not go to zero because somebody had a good harvest.

Against the stricter year-and-state specification (−0.05) the game looks steeper still
at every volume.

So the design is not a bad model of a market; it is a *deliberately impatient* one.
It compresses into a few hundred units of supply the kind of price destruction a real
commodity would need an enormous glut to produce, and it does so unevenly: wheat and
egg are almost perfectly inelastic at −0.04 no matter what you do, while melon and
strawberry fall off a cliff. That asymmetry is a design choice about which crops
punish volume, not an accident of calibration — and it is visible in the table above
long before you play a single game.

---

### Reproducing this

Attach `jessicali9530/honey-production` and run top to bottom. The game side needs no
data at all — it reads the pinned referee, quoted by symbol so it stays correct if the
line numbers move.

### Companion notebook

This one asks whether the game's price curve is *realistic*. A companion notebook,
[300 melons return 36% of face value](https://www.kaggle.com/code/xreina8/kaggriculture-300-melons-return-36-of-face-value),
asks what that curve *costs you inside the game* — how a single large sell order walks
its own price down, what the town absorbs each day, and why the shed caps everything.
Together they cover the same mechanism from the outside and the inside.

A third, [What decides rating once money stops mattering?](https://www.kaggle.com/code/xreina8/what-decides-rating-once-money-stops-mattering), turns to the
ladder itself: money stops separating agents above a rating of about 1000, so what
does?